In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl datasets openpyxl pandas
!pip install -q --no-deps peft accelerate bitsandbytes xformers


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import time
import torch
import pandas as pd

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.getenv('HF_TOKEN', '')

if not hf_token:
    raise ValueError('HF_TOKEN is missing. Add it in Colab Secrets or environment variables.')

print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab, go to Runtime > Change runtime type > T4 GPU.')

print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
PROMPT_CSV_PATH = '/content/drive/MyDrive/dataset_clean.csv'
ANSWER_XLSX_PATH = '/content/drive/MyDrive/Austrian Tax Law LLM Prompt Assignment.xlsx'
FT_JSON_PATH = '/content/drive/MyDrive/tax_finetune_dataset.json'
BASELINE_RESULTS_PATH = '/content/drive/MyDrive/baseline_results.csv'

ADAPTER_SAVE_PATH = '/content/drive/MyDrive/llama32_1b_tax_adapter'
PARTIAL_FT_RESULTS_PATH = '/content/drive/MyDrive/fine_tuned_results_partial.csv'
FINAL_FT_RESULTS_PATH = '/content/drive/MyDrive/fine_tuned_results.csv'
COMPARISON_RESULTS_PATH = '/content/drive/MyDrive/comparison_results.csv'

prompts_df = pd.read_csv(PROMPT_CSV_PATH)
prompts_df = prompts_df[['id', 'prompt']].dropna(subset=['prompt']).reset_index(drop=True)

answers_df = pd.read_excel(ANSWER_XLSX_PATH, sheet_name='Dataset')
answers_df = answers_df[['id', 'correct_answer', 'sources']].dropna(subset=['id'])
answers_df = answers_df.drop_duplicates(subset=['id']).reset_index(drop=True)

evaluation_df = prompts_df.merge(answers_df, on='id', how='left').reset_index(drop=True)

with open(FT_JSON_PATH, 'r', encoding='utf-8') as f:
    ft_examples = json.load(f)

print('Evaluation rows:', len(evaluation_df))
print('Fine-tuning rows:', len(ft_examples))
print('Missing gold answers after merge:', evaluation_df['correct_answer'].isna().sum())
evaluation_df.head(3)


In [ ]:
from huggingface_hub import login
login(token=hf_token)

from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-1B-Instruct-bnb-4bit',
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
)

print('Model and LoRA adapters are ready.')


In [ ]:
from datasets import Dataset

FT_SYSTEM_MSG = (
    'Du bist ein Experte fuer oesterreichisches Steuerrecht. '
    'Beantworte die Frage kurz, praezise und auf Deutsch. '
    'Nenne die relevante Gesetzesgrundlage, wenn moeglich.'
)

def build_chat_text(system_msg, user_msg, assistant_msg=None):
    text = (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n\n'
        f'{system_msg}<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n\n'
        f'{user_msg}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )
    if assistant_msg is not None:
        text += f'{assistant_msg}<|eot_id|>'
    return text

def format_example(example):
    return {'text': build_chat_text(FT_SYSTEM_MSG, example['prompt'], example['correct_answer'])}

formatted_dataset = Dataset.from_list(ft_examples).map(format_example)
print('Formatted rows:', len(formatted_dataset))
formatted_dataset[0]


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='/content/ft_model',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=5,
    fp16=True,
    logging_steps=5,
    save_strategy='no',
    eval_strategy='no',
    optim='adamw_8bit',
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    args=training_args,
)

print('Starting fine-tuning...')
t0 = time.time()
trainer.train()
print(f'Fine-tuning complete in {time.time() - t0:.0f}s.')


In [ ]:
trainer.model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print('Saved adapter to:', ADAPTER_SAVE_PATH)


In [ ]:
import torch

FastLanguageModel.for_inference(model)
tokenizer.padding_side = 'left'
tokenizer.pad_token = tokenizer.eos_token

model.eval()

ft_answers = []
BATCH_SIZE = 8
questions = evaluation_df['prompt'].tolist()
prompts = [build_chat_text(FT_SYSTEM_MSG, q) for q in questions]

print(f'Running fine-tuned inference on {len(questions)} questions (batch size {BATCH_SIZE})...')
t0 = time.time()

for i in range(0, len(prompts), BATCH_SIZE):
    batch = prompts[i : i + BATCH_SIZE]

    inputs = tokenizer(
        batch,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512,
    ).to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False,
        )

    input_len = inputs['input_ids'].shape[1]

    for out in outputs:
        ft_answers.append(tokenizer.decode(out[input_len:], skip_special_tokens=True).strip())

    done = min(i + BATCH_SIZE, len(prompts))
    if done % 50 == 0 or done == len(prompts):
        partial_df = pd.DataFrame({
            'id': evaluation_df['id'].iloc[:done].tolist(),
            'answer': ft_answers,
        })
        partial_df.to_csv(PARTIAL_FT_RESULTS_PATH, index=False)
        print(f'  {done}/{len(prompts)}')

print(f'FT inference done in {time.time() - t0:.0f}s.')


In [ ]:
ft_submission = pd.DataFrame({
    'id': evaluation_df['id'].tolist(),
    'answer': ft_answers,
})

ft_submission.to_csv(FINAL_FT_RESULTS_PATH, index=False)
print(f'Saved: {FINAL_FT_RESULTS_PATH} ({len(ft_submission)} rows)')
ft_submission.head()


In [ ]:
if os.path.exists(BASELINE_RESULTS_PATH):
    baseline_df = pd.read_csv(BASELINE_RESULTS_PATH)

    if 'baseline_answer' not in baseline_df.columns and 'answer' in baseline_df.columns:
        baseline_df = baseline_df.rename(columns={'answer': 'baseline_answer'})

    baseline_df = baseline_df[['id', 'baseline_answer']].drop_duplicates(subset=['id'])

    comparison_df = evaluation_df.merge(baseline_df, on='id', how='left')
    comparison_df = comparison_df.merge(ft_submission.rename(columns={'answer': 'tuned_answer'}), on='id', how='left')

    comparison_df = comparison_df[['id', 'prompt', 'correct_answer', 'sources', 'baseline_answer', 'tuned_answer']]
    comparison_df.to_csv(COMPARISON_RESULTS_PATH, index=False)

    print(f'Saved: {COMPARISON_RESULTS_PATH} ({len(comparison_df)} rows)')
    comparison_df.head()
else:
    print('baseline_results.csv not found in Drive. Skipping comparison file.')
